In [1]:
# load dungeon data

import json
import random
from pathlib import Path

data_path = Path("../datasets/dungeon_10k_4_8_3_5_mkr.jsonl")
with open(data_path) as f:
    data = [json.loads(line) for line in f]

# strip _id fields
for entry in data:
    if "_id" in entry:
        del entry["_id"]

# Split into train/eval sets
random.shuffle(data)
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
eval_data = data[split_idx:]

print(f"Train set: {len(train_data)} records")
print(f"Eval set: {len(eval_data)} records")

# print first data point
print(json.dumps(data[0], indent=2))

Train set: 8000 records
Eval set: 2000 records
{
  "door": 2,
  "key_color": "red",
  "corridor": [
    {
      "door_no": 7,
      "red_key": "diamonds",
      "green_key": "artifacts",
      "blue_key": "spellbooks"
    },
    {
      "monsters": [
        "dragon"
      ],
      "door_no": 6,
      "red_key": "gold",
      "blue_key": "spellbooks",
      "green_key": "spellbooks"
    },
    {
      "monsters": [
        "goblin",
        "wolf"
      ],
      "door_no": 1,
      "green_key": "diamonds",
      "blue_key": "artifacts",
      "red_key": "artifacts"
    },
    {
      "door_no": 4,
      "blue_key": "gemstones",
      "red_key": "gemstones",
      "green_key": "diamonds"
    },
    {
      "monsters": [
        "troll"
      ],
      "door_no": 0,
      "red_key": "gemstones",
      "blue_key": "artifacts",
      "green_key": "artifacts"
    },
    {
      "monsters": [
        "troll"
      ],
      "door_no": 5,
      "red_key": "spellbooks",
      "blue_key": "gold",

In [ ]:
from origami.pipeline import OrigamiPipeline, PipelineConfig
from origami.training import TableLogCallback, accuracy

config = PipelineConfig(
    d_model=192,
    n_heads=8,
    n_layers=6,
    d_ff=784,
    dropout=0.0,
    shuffle_keys=False,
    upscale_factor=1,
    batch_size=100,
    warmup_steps=1000,
    learning_rate=5e-4,
    eval_strategy="epoch",
    eval_epochs=5,
    eval_metrics={"acc": accuracy},
    eval_sample_size=100,
    target_key="treasure",
    use_grammar_constraints=True,
)

pipeline = OrigamiPipeline(config)
pipeline.fit(
    train_data, eval_data=eval_data, callbacks=[TableLogCallback(print_every=50)], epochs=250
)

| step: 50 | epoch: 0 | lr: 2.50e-05 | batch_dt: 150ms | loss: 2.3263 |
| step: 100 | epoch: 1 | lr: 5.00e-05 | batch_dt: 139ms | loss: 1.3713 |
| step: 150 | epoch: 1 | lr: 7.50e-05 | batch_dt: 147ms | loss: 1.0767 |
| step: 200 | epoch: 2 | lr: 1.00e-04 | batch_dt: 144ms | loss: 0.9622 |
| step: 250 | epoch: 3 | lr: 1.25e-04 | batch_dt: 155ms | loss: 0.8626 |
| step: 300 | epoch: 3 | lr: 1.50e-04 | batch_dt: 118ms | loss: 0.8350 |
| step: 350 | epoch: 4 | lr: 1.75e-04 | batch_dt: 107ms | loss: 0.8202 |
| step: 400 | epoch: 4 | lr: 2.00e-04 | batch_dt: 113ms | loss: 0.8199 |
| step: 450 | epoch: 5 | lr: 2.25e-04 | batch_dt: 110ms | loss: 0.8123 | val_acc: 0.1900 | val_loss: 0.8246 |
| step: 500 | epoch: 6 | lr: 2.50e-04 | batch_dt: 157ms | loss: 0.7979 |
| step: 550 | epoch: 6 | lr: 2.75e-04 | batch_dt: 148ms | loss: 0.7880 |
| step: 600 | epoch: 7 | lr: 3.00e-04 | batch_dt: 125ms | loss: 0.7708 |
| step: 650 | epoch: 8 | lr: 3.25e-04 | batch_dt: 105ms | loss: 0.7664 |
| step: 700 | e

In [ ]:
pipeline.save("dungeon_pipeline.pt")

In [ ]:
from origami.pipeline import OrigamiPipeline

pipeline = OrigamiPipeline.load("dungeon_pipeline.pt")

In [ ]:
from origami.training import accuracy

pipeline.evaluate(eval_data, metrics={"acc": accuracy})

In [ ]:
doc = pipeline.generate(1)[0]

print(json.dumps(doc, indent=2))